In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os, shutil

drive_training_data = "/content/drive/MyDrive/Fun_projects/Detection_objets_sur_imagerie_satellite/data_lake/xView/training_data"
local_training_data = "/content/xView_local/training_data"

os.makedirs(f"{local_training_data}/images", exist_ok=True)
os.makedirs(f"{local_training_data}/labels", exist_ok=True)

for split in ["train", "val"]:
    shutil.copytree(f"{drive_training_data}/images/{split}", f"{local_training_data}/images/{split}", dirs_exist_ok=True)
    shutil.copytree(f"{drive_training_data}/labels_merged/{split}", f"{local_training_data}/labels/{split}", dirs_exist_ok=True)

shutil.copy(f"{drive_training_data}/data_merged.yaml", f"{local_training_data}/data_merged.yaml")
local_yaml_merged_path = f"{local_training_data}/data_merged.yaml"
print("Dataset reconstruit.")

Dataset reconstruit.


In [4]:
!pip install -q ultralytics
import torch
print("GPU disponible :", torch.cuda.is_available())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 10.2 MB/s eta 0:00:00
GPU disponible : True


In [5]:
from ultralytics import YOLO

run_name = "xview_yolov8s_merged_v3"
drive_runs_dir = "/content/drive/MyDrive/Fun_projects/Detection_objets_sur_imagerie_satellite/data_lake/xView/runs"

def backup_to_drive(trainer):
    dest = f"{drive_runs_dir}/{run_name}"
    try:
        shutil.copytree(str(trainer.save_dir), dest, dirs_exist_ok=True)
        print(f"[backup] epoch {trainer.epoch} — dossier complet sauvegardé sur Drive")
    except Exception as e:
        print(f"[backup] échec à l'epoch {trainer.epoch} : {e}")

model = YOLO(f"{drive_runs_dir}/xview_yolov8s_merged_v2/last.pt")
model.add_callback("on_fit_epoch_end", backup_to_drive)

results = model.train(
    data=local_yaml_merged_path,
    epochs=63,
    imgsz=640,
    batch=16,
    patience=30,
    save_period=1,
    project="/content/xView_local/runs",
    name=run_name
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.160 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/xView_local/training_data/data_merged.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=63, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torch

In [6]:
import shutil, os, yaml

final_run_dir = str(model.trainer.save_dir)
dest_final = f"{drive_runs_dir}/{run_name}"

shutil.copytree(final_run_dir, dest_final, dirs_exist_ok=True)
print("Run complet sauvegardé sur Drive :", dest_final)

best_weights = f"{final_run_dir}/weights/best.pt"
metrics = model.val(data=local_yaml_merged_path, split="val")

summary = {
    "mAP50": float(metrics.box.map50),
    "mAP50-95": float(metrics.box.map),
    "precision": float(metrics.box.mp),
    "recall": float(metrics.box.mr),
    "epochs_completed": model.trainer.epoch + 1,
}

with open(f"{final_run_dir}/final_summary.yaml", "w") as f:
    yaml.dump(summary, f, sort_keys=False)
shutil.copy(f"{final_run_dir}/final_summary.yaml", f"{dest_final}/final_summary.yaml")

print("\n=== RÉSUMÉ FINAL ===")
for k, v in summary.items():
    print(f"{k} : {v}")

Run complet sauvegardé sur Drive : /content/drive/MyDrive/Fun_projects/Detection_objets_sur_imagerie_satellite/data_lake/xView/runs/xview_yolov8s_merged_v3
Ultralytics 8.4.160 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 11,136,807 parameters, 0 gradients, 28.5 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 34.0±24.2 MB/s, size: 33.1 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/xView_local/training_data/labels/val.cache... 1878 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1878/1878 437.6Mit/s 0.0s
WARNING ⚠️ Dataset images contain up to 950 objects (val=950), but max_det=300. This mismatch can cap recall and produce invalid validation metrics. Raising it may increase validation cost but cannot increase model or export capacity, which may cap recall. Setting max_det=950 to match the 